# Amazon Review Helpfulness Classification

## Project Objective

This project predicts whether an Amazon Fine Food Review is helpful based only on its written content.

A review is labeled **Helpful (1)** when at least 50% of its recorded helpfulness votes are positive, and **Not Helpful (0)** otherwise. Reviews with no helpfulness votes are excluded because a helpfulness ratio cannot be calculated.

The project compares two NLP representations — **TF-IDF** and **Word2Vec** — across multiple machine-learning classifiers. Because the target is class-imbalanced, model selection emphasizes **Macro F1** alongside accuracy, precision, recall, and confusion matrices.

In [7]:
# Core libraries
import sys
import numpy as np
import pandas as pd
import scipy
import gensim

# NLP
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

# Model selection
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    GridSearchCV,
    StratifiedKFold,
    cross_validate
)

# Models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier

# Pipelines and metrics
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    make_scorer,
    f1_score,
    precision_score,
    recall_score
)

## 1. Dataset Loading and Initial Inspection

In [9]:
df=pd.read_csv('Reviews.csv')
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [43]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [44]:
df.shape

(568454, 10)

In [45]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09


In [11]:
#Missing values
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

## 2. Target Construction

Reviews without helpfulness votes are removed because their helpfulness ratio is undefined.

For the remaining reviews:

- **Helpful (1):** helpfulness ratio ≥ 0.50
- **Not Helpful (0):** helpfulness ratio < 0.50

In [13]:
#Remove the rows with 0 helpfulness votes
df = df[df["HelpfulnessDenominator"] > 0].copy()

In [15]:
# Add the helpfulness ratio column
df["helpfulness_ratio"] = df["HelpfulnessNumerator"] / df["HelpfulnessDenominator"]


In [17]:
#Creating the Target Variable
df["Is_Helpful"] = np.where(df["helpfulness_ratio"] >= 0.5, 1, 0)
df["Is_Helpful"].value_counts()

Is_Helpful
1    248289
0     50113
Name: count, dtype: int64

In [21]:
#Combine the summary and the text column to avoid multiple runs of word2vec and TF-IDF
df["Review_Text"] = df["Summary"].fillna("") + " " + df["Text"].fillna("")
df.head(3)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,helpfulness_ratio,Is_Helpful,Review_Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,1.0,1,Good Quality Dog Food I have bought several of...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,1.0,1,"""Delight"" says it all This is a confection tha..."
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...,1.0,1,Cough Medicine If you are looking for the secr...


## 3. Data Quality and Duplicate-Leakage Audit

Exact duplicate review text can appear in both training and test sets after a random split, allowing the model to see effectively the same review during training and evaluation.

To reduce this source of leakage:

1. Exact review text with conflicting target labels is removed.
2. Only one copy of each remaining exact review text is retained.

In [23]:
duplicate_reviews = df.duplicated(
    subset=["Review_Text"]
).sum()

print("Duplicate review texts:", duplicate_reviews)

Duplicate review texts: 88325


In [25]:
# Identify review texts that have more than one target label
conflicting_texts = (
    df.groupby("Review_Text")["Is_Helpful"]
      .nunique()
)

conflicting_texts = conflicting_texts[
    conflicting_texts > 1
].index

print(
    "Conflicting review texts to remove:",
    len(conflicting_texts)
)

Conflicting review texts to remove: 117


In [27]:
# Remove all reviews whose exact text has conflicting target labels
df_clean = df[
    ~df["Review_Text"].isin(conflicting_texts)
].copy()

# Keep only one instance of every remaining exact review text
df_clean = df_clean.drop_duplicates(
    subset=["Review_Text"]
).reset_index(drop=True)

print("Original rows:", len(df))
print("Rows after cleaning:", len(df_clean))
print(
    "Remaining duplicate review texts:",
    df_clean.duplicated(subset=["Review_Text"]).sum()
)

Original rows: 298402
Rows after cleaning: 209960
Remaining duplicate review texts: 0


In [29]:
print(df_clean["Is_Helpful"].value_counts())

print("\nClass proportions:")
print(
    df_clean["Is_Helpful"].value_counts(
        normalize=True
    )
)

Is_Helpful
1    176467
0     33493
Name: count, dtype: int64

Class proportions:
Is_Helpful
1    0.840479
0    0.159521
Name: proportion, dtype: float64


## 4. Text Preprocessing

In [31]:
#Remove negation words from the stopwords list
negation_words = {
    "not",
    "no",
    "never",
    "cannot",
    "cant",
    "couldnt",
    "nothing"
}

stop_words = STOPWORDS.difference(
    negation_words
)

print("Number of stopwords:", len(stop_words))

Number of stopwords: 330


In [33]:
# Tokenize cleaned review text for Word2Vec
def tokenize_review(text):
    return simple_preprocess(str(text))

df_clean["Tokens"] = df_clean["Review_Text"].apply(tokenize_review)

df_clean[["Review_Text", "Tokens"]].head()

,Review_Text,Tokens
0,Good Quality Dog Food I have bought several of...,"[good, quality, dog, food, have, bought, sever..."
1,"""Delight"" says it all This is a confection tha...","[delight, says, it, all, this, is, confection,..."
2,Cough Medicine If you are looking for the secr...,"[cough, medicine, if, you, are, looking, for, ..."
3,Yay Barley Right now I'm mostly just sprouting...,"[yay, barley, right, now, mostly, just, sprout..."
4,The Best Hot Sauce in the World I don't know i...,"[the, best, hot, sauce, in, the, world, don, k..."


## 5. Train/Test Split

The cleaned dataset is split using an 80/20 stratified split so that the Helpful/Not Helpful class distribution is preserved in both sets.

The same row split is used for both TF-IDF and Word2Vec representations to make model comparisons consistent.

In [35]:
# Features for the two NLP representations
X_text = df_clean["Review_Text"]   # Used for TF-IDF
X_tokens = df_clean["Tokens"]      # Used for Word2Vec
y = df_clean["Is_Helpful"]         # Target

In [37]:
(
    X_train_text,
    X_test_text,
    X_train_tokens,
    X_test_tokens,
    y_train,
    y_test
) = train_test_split(
    X_text,
    X_tokens,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y # Helps with class imbalance
)

print("Training rows:", len(y_train))
print("Test rows:", len(y_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Training rows: 167968
Test rows: 41992

Training class distribution:
Is_Helpful
1    0.840482
0    0.159518
Name: proportion, dtype: float64

Test class distribution:
Is_Helpful
1    0.84047
0    0.15953
Name: proportion, dtype: float64


### Majority-Class Baseline

Because approximately 84% of reviews are labeled Helpful, a model that predicts Helpful for every review would already achieve roughly 84% accuracy. This baseline demonstrates why accuracy alone is not sufficient for evaluating this task.

In [39]:
dummy_model = DummyClassifier(
    strategy="most_frequent"
)

dummy_model.fit(
    X_train_text,
    y_train
)

dummy_pred = dummy_model.predict(
    X_test_text
)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test, dummy_pred),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        dummy_pred,
        digits=3,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        dummy_pred
    )
)

Accuracy: 0.8405

Classification Report:
              precision    recall  f1-score   support

           0      0.000     0.000     0.000      6699
           1      0.840     1.000     0.913     35293

    accuracy                          0.840     41992
   macro avg      0.420     0.500     0.457     41992
weighted avg      0.706     0.840     0.768     41992

Confusion Matrix:
[[    0  6699]
 [    0 35293]]


## 6. TF-IDF Experiments

TF-IDF converts review text into a sparse 10,000-feature representation. Logistic Regression, Random Forest, Linear SVM, and SGDClassifier are evaluated using the same train/test split.

In [41]:
#TF-IDF Model
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words=sorted(stop_words)
)


In [45]:
# Learn the TF-IDF vocabulary from training data only
X_train_tfidf = tfidf.fit_transform(X_train_text)

# Apply the fitted TF-IDF transformation to unseen test data
X_test_tfidf = tfidf.transform(X_test_text)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (167968, 10000)
Test TF-IDF shape: (41992, 10000)


In [46]:
# TF-IDF Logistic Regression baseline
log_tfidf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_tfidf.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [49]:
# Evaluate TF-IDF Logistic Regression
tfidf_log_pred = log_tfidf.predict(X_test_tfidf)

print(
    "Accuracy:",
    round(accuracy_score(y_test, tfidf_log_pred), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        tfidf_log_pred,
        digits=3
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        tfidf_log_pred
    )
)

Accuracy: 0.6957

Classification Report:
              precision    recall  f1-score   support

           0      0.291     0.633     0.399      6699
           1      0.910     0.708     0.796     35293

    accuracy                          0.696     41992
   macro avg      0.601     0.670     0.598     41992
weighted avg      0.812     0.696     0.733     41992

Confusion Matrix:
[[ 4239  2460]
 [10319 24974]]


In [108]:
# Hyperparameter search space for TF-IDF Random Forest
tfidf_rf_params = {
    "n_estimators": [30, 50, 75, 100],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [2, 5, 10],
    "max_features": ["sqrt", "log2"]
}


In [110]:
# Base Random Forest for TF-IDF tuning
rf_tfidf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [112]:
# Tune TF-IDF Random Forest
rf_tfidf_search = RandomizedSearchCV(
    estimator=rf_tfidf_base,
    param_distributions=tfidf_rf_params,
    n_iter=10,
    scoring="f1_macro",
    cv=3,
    verbose=0,
    random_state=42,
    n_jobs=-1
)

rf_tfidf_search.fit(
    X_train_tfidf,
    y_train
)

print(
    "Best CV Macro F1:",
    round(rf_tfidf_search.best_score_, 4)
)

print(
    "Best parameters:",
    rf_tfidf_search.best_params_
)

Best CV Macro F1: 0.6187
Best parameters: {'n_estimators': 30, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}


In [114]:
# Select and evaluate the best TF-IDF Random Forest
best_rf_tfidf = rf_tfidf_search.best_estimator_

rf_tfidf_pred = best_rf_tfidf.predict(
    X_test_tfidf
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            rf_tfidf_pred
        ),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_tfidf_pred,
        digits=3
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_tfidf_pred
    )
)

Accuracy: 0.787

Classification Report:
              precision    recall  f1-score   support

           0      0.344     0.371     0.357      6699
           1      0.879     0.866     0.872     35293

    accuracy                          0.787     41992
   macro avg      0.612     0.619     0.615     41992
weighted avg      0.794     0.787     0.790     41992

Confusion Matrix:
[[ 2487  4212]
 [ 4734 30559]]


In [51]:


svm_tfidf = LinearSVC(
    class_weight="balanced",
    dual="auto",
    random_state=42
)

svm_tfidf.fit(
    X_train_tfidf,
    y_train
)

svm_tfidf_pred = svm_tfidf.predict(
    X_test_tfidf
)

print(
    classification_report(
        y_test,
        svm_tfidf_pred,
        digits=3
    )
)

print(
    confusion_matrix(
        y_test,
        svm_tfidf_pred
    )
)

              precision    recall  f1-score   support

           0      0.281     0.609     0.384      6699
           1      0.905     0.704     0.792     35293

    accuracy                          0.689     41992
   macro avg      0.593     0.656     0.588     41992
weighted avg      0.805     0.689     0.727     41992

[[ 4081  2618]
 [10455 24838]]


In [52]:
# SGDClassifier baseline


sgd_tfidf = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

sgd_tfidf.fit(
    X_train_tfidf,
    y_train
)

sgd_tfidf_pred = sgd_tfidf.predict(
    X_test_tfidf
)

print(
    classification_report(
        y_test,
        sgd_tfidf_pred,
        digits=3
    )
)

print(
    confusion_matrix(
        y_test,
        sgd_tfidf_pred
    )
)

              precision    recall  f1-score   support

           0      0.289     0.652     0.401      6699
           1      0.913     0.696     0.790     35293

    accuracy                          0.689     41992
   macro avg      0.601     0.674     0.595     41992
weighted avg      0.814     0.689     0.728     41992

[[ 4369  2330]
 [10738 24555]]


### TF-IDF + SGDClassifier Hyperparameter Tuning

In [198]:
# Base SGD model for TF-IDF tuning

sgd_tfidf_base = SGDClassifier(
    loss="log_loss",
    max_iter=2000,
    tol=1e-3,
    random_state=42,
    n_jobs=1
)

# Focused hyperparameter grid
sgd_param_grid = {
    "alpha": [
        0.00001,
        0.0001,
        0.001
    ],

    "penalty": [
        "l2",
        "l1"
    ],

    "class_weight": [
        None,
        "balanced",
        {0: 1.5, 1: 1},
        {0: 2, 1: 1},
        {0: 2.5, 1: 1}
    ]
}

In [204]:
# Grid Search for TF-IDF + SGDClassifier

sgd_tfidf_grid = GridSearchCV(
    estimator=sgd_tfidf_base,
    param_grid=sgd_param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

sgd_tfidf_grid.fit(
    X_train_tfidf,
    y_train
)

print(
    "\nBest CV Macro F1:",
    round(sgd_tfidf_grid.best_score_, 4)
)

print(
    "Best Parameters:",
    sgd_tfidf_grid.best_params_
)

Fitting 3 folds for each of 30 candidates, totalling 90 fits

Best CV Macro F1: 0.626
Best Parameters: {'alpha': 1e-05, 'class_weight': {0: 2.5, 1: 1}, 'penalty': 'l1'}


In [206]:
# Best tuned TF-IDF + SGDClassifier

best_sgd_tfidf = sgd_tfidf_grid.best_estimator_

sgd_tfidf_tuned_pred = best_sgd_tfidf.predict(
    X_test_tfidf
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            sgd_tfidf_tuned_pred
        ),
        4
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        sgd_tfidf_tuned_pred,
        digits=3
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        sgd_tfidf_tuned_pred
    )
)

Accuracy: 0.8041

Classification Report:
              precision    recall  f1-score   support

           0      0.377     0.349     0.363      6699
           1      0.878     0.890     0.884     35293

    accuracy                          0.804     41992
   macro avg      0.628     0.620     0.623     41992
weighted avg      0.798     0.804     0.801     41992

Confusion Matrix:
[[ 2340  4359]
 [ 3868 31425]]


### TF-IDF Robustness Check — Tuned SGDClassifier

The tuned TF-IDF + SGDClassifier produced the strongest TF-IDF performance so far, achieving a **Macro F1 of 0.623** on the held-out test set.

Before moving to the Word2Vec experiments, an additional robustness check is performed to determine whether this performance remains stable across different subsets of the cleaned dataset.

The winning SGDClassifier hyperparameters identified through GridSearchCV are kept fixed:

- **Loss:** `log_loss`
- **Alpha:** `1e-5`
- **Penalty:** `l1`
- **Class Weight:** `{0: 2.5, 1: 1}`

A **stratified 5-fold cross-validation** is then performed using a scikit-learn Pipeline containing both TF-IDF and the tuned SGDClassifier.

Placing TF-IDF inside the Pipeline ensures that the TF-IDF vocabulary is learned independently within each training fold rather than being fitted on data from the corresponding validation fold.

The purpose of this step is not to perform additional hyperparameter tuning, but to evaluate the **stability and consistency** of the selected TF-IDF configuration before comparing it with the Word2Vec-based models.

In [81]:
# Final TF-IDF + SGD pipeline using the winning hyperparameters

final_tfidf_sgd = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=10000,
            stop_words=list(stop_words)
        )
    ),
    (
        "sgd",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            penalty="l1",
            class_weight={0: 2.5, 1: 1},
            max_iter=2000,
            tol=1e-3,
            random_state=42,
            n_jobs=1
        )
    )
])

In [83]:
# Stratified 5-fold cross-validation

final_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "class0_f1": make_scorer(
        f1_score,
        pos_label=0
    ),
    "class0_precision": make_scorer(
        precision_score,
        pos_label=0
    ),
    "class0_recall": make_scorer(
        recall_score,
        pos_label=0
    )
}

final_cv_results = cross_validate(
    final_tfidf_sgd,
    X_text,
    y,
    cv=final_cv,
    scoring=scoring,
    n_jobs=-1
)

print("5-fold cross-validation completed.")

5-fold cross-validation completed.


In [85]:
# Display fold-by-fold validation results

cv_summary = pd.DataFrame({
    "Accuracy": final_cv_results["test_accuracy"],
    "Macro F1": final_cv_results["test_macro_f1"],
    "Class 0 Precision": final_cv_results["test_class0_precision"],
    "Class 0 Recall": final_cv_results["test_class0_recall"],
    "Class 0 F1": final_cv_results["test_class0_f1"]
})

print("Fold-by-Fold Results:\n")
print(cv_summary.round(3))

print("\nAverage 5-Fold Results:\n")

for column in cv_summary.columns:
    print(
        f"{column}: "
        f"{cv_summary[column].mean():.3f} "
        f"(± {cv_summary[column].std():.3f})"
    )

Fold-by-Fold Results:

   Accuracy  Macro F1  Class 0 Precision  Class 0 Recall  Class 0 F1
0     0.806     0.624              0.381           0.346       0.363
1     0.809     0.624              0.388           0.337       0.360
2     0.809     0.629              0.390           0.353       0.370
3     0.815     0.622              0.400           0.313       0.351
4     0.809     0.634              0.395           0.369       0.382

Average 5-Fold Results:

Accuracy: 0.810 (± 0.003)
Macro F1: 0.627 (± 0.005)
Class 0 Precision: 0.391 (± 0.007)
Class 0 Recall: 0.344 (± 0.021)
Class 0 F1: 0.365 (± 0.011)


#### TF-IDF Robustness Check Results

The tuned TF-IDF + SGDClassifier demonstrated stable performance across the five stratified folds:

- **Average Accuracy:** 0.810 ± 0.003
- **Average Macro F1:** 0.627 ± 0.005
- **Average Class 0 Precision:** 0.391 ± 0.007
- **Average Class 0 Recall:** 0.344 ± 0.021
- **Average Class 0 F1:** 0.365 ± 0.011

The small variation in Macro F1 across folds suggests that the TF-IDF + tuned SGDClassifier configuration performs consistently across different subsets of the cleaned dataset.

These results establish a strong TF-IDF benchmark that can now be compared with the Word2Vec-based models in the following section.

## 7. Word2Vec Experiments

Word2Vec learns a 200-dimensional dense representation from the training reviews. Each review is represented by the mean of the vectors for words contained in the trained vocabulary.

The same Logistic Regression, Random Forest, Linear SVM, and SGDClassifier families are evaluated for comparison with TF-IDF.

In [55]:
# Train Word2Vec on training reviews only
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=42
)

print("Vocabulary size:", len(w2v_model.wv))
print("Embedding dimensions:", w2v_model.vector_size)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Vocabulary size: 46100
Embedding dimensions: 200


In [124]:
print("Vocabulary size:", len(w2v_model.wv))

print(
    "All Word2Vec values valid:",
    np.isfinite(w2v_model.wv.vectors).all()
)

print(
    "Word2Vec matrix shape:",
    w2v_model.wv.vectors.shape
)

Vocabulary size: 46100
All Word2Vec values valid: True
Word2Vec matrix shape: (46100, 200)


In [58]:
# Function to convert one tokenized review into one 200-dimensional vector
def average_word2vec(tokens, keyed_vectors, vector_size=200):
    vectors = [
        keyed_vectors[word]
        for word in tokens
        if word in keyed_vectors
    ]

    if vectors:
        return np.mean(vectors, axis=0)

    return np.zeros(vector_size)

In [60]:
# Create averaged Word2Vec vectors for training reviews
X_train_w2v = np.array([
    average_word2vec(
        tokens,
        w2v_model.wv,
        vector_size=200
    )
    for tokens in X_train_tokens
])

print("Training Word2Vec shape:", X_train_w2v.shape)

Training Word2Vec shape: (167968, 200)


In [61]:
# Create averaged Word2Vec vectors for unseen test reviews
X_test_w2v = np.array([
    average_word2vec(
        tokens,
        w2v_model.wv,
        vector_size=200
    )
    for tokens in X_test_tokens
])

print("Test Word2Vec shape:", X_test_w2v.shape)

Test Word2Vec shape: (41992, 200)


In [64]:
# Validate the generated review vectors
print(
    "Training vectors valid:",
    np.isfinite(X_train_w2v).all()
)

print(
    "Test vectors valid:",
    np.isfinite(X_test_w2v).all()
)

Training vectors valid: True
Test vectors valid: True


In [66]:
# Word2Vec Logistic Regression baseline
log_w2v = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_w2v.fit(X_train_w2v, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [68]:
# Evaluate Word2Vec Logistic Regression
w2v_log_pred = log_w2v.predict(X_test_w2v)

print(
    "Accuracy:",
    round(accuracy_score(y_test, w2v_log_pred), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        w2v_log_pred,
        digits=3
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        w2v_log_pred
    )
)

Accuracy: 0.6736

Classification Report:
              precision    recall  f1-score   support

           0      0.277     0.648     0.388      6699
           1      0.910     0.678     0.777     35293

    accuracy                          0.674     41992
   macro avg      0.593     0.663     0.583     41992
weighted avg      0.809     0.674     0.715     41992

Confusion Matrix:
[[ 4338  2361]
 [11347 23946]]


In [146]:
# Word2Vec Random Forest baseline
rf_w2v = RandomForestClassifier(
    n_estimators=100,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_w2v.fit(
    X_train_w2v,
    y_train
)

RandomForestClassifier(class_weight='balanced', max_depth=30,
                       min_samples_leaf=2, min_samples_split=5, n_jobs=-1,
                       random_state=42)

In [148]:
# Evaluate Word2Vec Random Forest baseline
rf_w2v_pred = rf_w2v.predict(
    X_test_w2v
)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test, rf_w2v_pred),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_w2v_pred,
        digits=3
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_w2v_pred
    )
)

Accuracy: 0.8401

Classification Report:
              precision    recall  f1-score   support

           0      0.459     0.013     0.025      6699
           1      0.842     0.997     0.913     35293

    accuracy                          0.840     41992
   macro avg      0.651     0.505     0.469     41992
weighted avg      0.781     0.840     0.771     41992

Confusion Matrix:
[[   85  6614]
 [  100 35193]]


In [174]:
# Hyperparameter search space for Word2Vec Random Forest

w2v_rf_params = {
    "n_estimators": [100, 150, 200],
    "max_depth": [15, 25, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5],
    "class_weight": [
        "balanced",
        "balanced_subsample",
        {0: 3, 1: 1},
        {0: 5, 1: 1}
    ]
}

In [176]:
# Base Random Forest for Word2Vec hyperparameter tuning

rf_w2v_base = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=5,
    random_state=42,
    n_jobs=1
)

In [178]:
# Smaller stratified sample for fast hyperparameter tuning

X_w2v_tune, _, y_w2v_tune, _ = train_test_split(
    X_train_w2v,
    y_train,
    train_size=20000,
    random_state=42,
    stratify=y_train
)

print("Tuning sample shape:", X_w2v_tune.shape)

print("\nClass distribution:")
print(y_w2v_tune.value_counts(normalize=True))

Tuning sample shape: (20000, 200)

Class distribution:
Is_Helpful
1    0.8405
0    0.1595
Name: proportion, dtype: float64


In [180]:
# Fast focused tuning

rf_w2v_search = RandomizedSearchCV(
    estimator=rf_w2v_base,
    param_distributions=w2v_rf_params,
    n_iter=4,
    scoring="f1_macro",
    cv=2,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_w2v_search.fit(
    X_w2v_tune,
    y_w2v_tune
)

print(
    "\nBest CV Macro F1:",
    round(rf_w2v_search.best_score_, 4)
)

print(
    "Best parameters:",
    rf_w2v_search.best_params_
)

Fitting 2 folds for each of 4 candidates, totalling 8 fits

Best CV Macro F1: 0.4836
Best parameters: {'n_estimators': 100, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': 15, 'class_weight': {0: 5, 1: 1}}


In [184]:
# Take the best Random Forest configuration found during tuning
best_rf_w2v = rf_w2v_search.best_estimator_

# Use all CPU cores for the final full-data training
best_rf_w2v.set_params(n_jobs=-1)

# Retrain the winning model on ALL training data
best_rf_w2v.fit(
    X_train_w2v,
    y_train
)

print(
    "Final Word2Vec Random Forest trained on full training set."
)

Final Word2Vec Random Forest trained on full training set.


In [186]:
rf_w2v_tuned_pred = best_rf_w2v.predict(
    X_test_w2v
)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test, rf_w2v_tuned_pred),
        4
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_w2v_tuned_pred,
        digits=3
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        rf_w2v_tuned_pred
    )
)

Accuracy: 0.8261

Classification Report:
              precision    recall  f1-score   support

           0      0.396     0.171     0.238      6699
           1      0.858     0.951     0.902     35293

    accuracy                          0.826     41992
   macro avg      0.627     0.561     0.570     41992
weighted avg      0.784     0.826     0.796     41992

Confusion Matrix:
[[ 1143  5556]
 [ 1747 33546]]


In [70]:
# Linear SVM Baseline
svm_w2v = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_w2v.fit(
    X_train_w2v,
    y_train
)

svm_w2v_pred = svm_w2v.predict(
    X_test_w2v
)

print(
    classification_report(
        y_test,
        svm_w2v_pred,
        digits=3
    )
)

print(
    confusion_matrix(
        y_test,
        svm_w2v_pred
    )
)

C:\Users\yatha\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


              precision    recall  f1-score   support

           0      0.277     0.644     0.387      6699
           1      0.910     0.681     0.779     35293

    accuracy                          0.675     41992
   macro avg      0.593     0.662     0.583     41992
weighted avg      0.809     0.675     0.716     41992

[[ 4311  2388]
 [11267 24026]]


In [72]:
# SGDClassifier baseline
sgd_w2v = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

sgd_w2v.fit(
    X_train_w2v,
    y_train
)

sgd_w2v_pred = sgd_w2v.predict(
    X_test_w2v
)

print(
    classification_report(
        y_test,
        sgd_w2v_pred,
        digits=3
    )
)

print(
    confusion_matrix(
        y_test,
        sgd_w2v_pred
    )
)

              precision    recall  f1-score   support

           0      0.276     0.641     0.386      6699
           1      0.909     0.681     0.778     35293

    accuracy                          0.674     41992
   macro avg      0.592     0.661     0.582     41992
weighted avg      0.808     0.674     0.716     41992

[[ 4292  2407]
 [11273 24020]]


## Final Model Selection and Conclusion

After comparing TF-IDF and Word2Vec representations across Logistic Regression, Random Forest, Linear SVM, and SGDClassifier models, the **TF-IDF + tuned SGDClassifier** was selected as the final model.

GridSearchCV identified the following SGDClassifier configuration:

- `loss = "log_loss"`
- `alpha = 1e-5`
- `penalty = "l1"`
- `class_weight = {0: 2.5, 1: 1}`

On the held-out test set, the tuned model achieved:

- **Accuracy:** 0.804
- **Macro F1:** 0.623
- **Class 0 F1:** 0.363

A subsequent stratified 5-fold robustness check produced an average **Macro F1 of 0.627 ± 0.005**, indicating consistent performance across different subsets of the cleaned dataset.

The final model therefore uses a **10,000-feature TF-IDF representation with a tuned SGDClassifier**. This pipeline will be used for deployment because it provides the strongest balanced classification performance observed in the project while remaining computationally efficient and suitable for real-time inference.

## Model Export for Deployment

The selected TF-IDF + SGDClassifier pipeline is retrained on the complete cleaned dataset and saved as a single Joblib artifact for deployment.

In [77]:
import joblib

In [87]:
# Train the final production pipeline on all cleaned review data

final_tfidf_sgd.fit(
    X_text,
    y
)

print("Final production model trained on all cleaned reviews.")
print("Training rows:", len(y))

Final production model trained on all cleaned reviews.
Training rows: 209960


In [89]:
# Save the complete TF-IDF + SGD pipeline

model_filename = "amazon_helpfulness_tfidf_sgd.joblib"

joblib.dump(
    final_tfidf_sgd,
    model_filename,
    compress=3
)

print("Model saved as:", model_filename)

Model saved as: amazon_helpfulness_tfidf_sgd.joblib


In [91]:
# Reload the saved model to verify the artifact

loaded_model = joblib.load(
    "amazon_helpfulness_tfidf_sgd.joblib"
)

print("Model loaded successfully.")

Model loaded successfully.


In [93]:
sample_reviews = [
    "Excellent product. It arrived quickly and worked exactly as expected.",
    "Terrible quality and a complete waste of money. I would not recommend it."
]

sample_predictions = loaded_model.predict(
    sample_reviews
)

sample_probabilities = loaded_model.predict_proba(
    sample_reviews
)

for review, prediction, probability in zip(
    sample_reviews,
    sample_predictions,
    sample_probabilities
):
    print("\nReview:")
    print(review)

    print(
        "Prediction:",
        "Helpful" if prediction == 1 else "Not Helpful"
    )

    print(
        "Probability of Not Helpful:",
        round(probability[0], 3)
    )

    print(
        "Probability of Helpful:",
        round(probability[1], 3)
    )


Review:
Excellent product. It arrived quickly and worked exactly as expected.
Prediction: Helpful
Probability of Not Helpful: 0.266
Probability of Helpful: 0.734

Review:
Terrible quality and a complete waste of money. I would not recommend it.
Prediction: Not Helpful
Probability of Not Helpful: 0.698
Probability of Helpful: 0.302


In [95]:
print(loaded_model)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000,
                                 stop_words=['do', 'sincere', 'now', 'be',
                                             'everyone', 'thence', 'each',
                                             'over', 'really', 'am', 'whence',
                                             'three', 'behind', 'using',
                                             'their', 'unless', 'he', 'why',
                                             'whether', 'last', 'whither',
                                             'although', 'amoungst', 'should',
                                             'are', 'hereupon', 'out', 'anyhow',
                                             'more', 'becomes', ...])),
                ('sgd',
                 SGDClassifier(alpha=1e-05, class_weight={0: 2.5, 1: 1},
                               loss='log_loss', max_iter=2000, n_jobs=1,
                               penalty='l1', random_state=42))])


In [97]:
print(
    loaded_model.named_steps["tfidf"].max_features
)

print(
    loaded_model.named_steps["sgd"].get_params()
)

10000
{'alpha': 1e-05, 'average': False, 'class_weight': {0: 2.5, 1: 1}, 'early_stopping': False, 'epsilon': 0.1, 'eta0': 0.0, 'fit_intercept': True, 'l1_ratio': 0.15, 'learning_rate': 'optimal', 'loss': 'log_loss', 'max_iter': 2000, 'n_iter_no_change': 5, 'n_jobs': 1, 'penalty': 'l1', 'power_t': 0.5, 'random_state': 42, 'shuffle': True, 'tol': 0.001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}


In [99]:
import joblib

model = joblib.load(
    "amazon_helpfulness_tfidf_sgd.joblib"
)

tfidf_model = model.named_steps["tfidf"]
sgd_model = model.named_steps["sgd"]

feature_names = tfidf_model.get_feature_names_out()

feature_weights = dict(
    zip(
        feature_names,
        sgd_model.coef_[0]
    )
)

words_to_check = [
    "not",
    "no",
    "never",
    "great",
    "good",
    "bad",
    "terrible",
    "excellent",
    "disappointed",
    "waste"
]

for word in words_to_check:
    if word in feature_weights:
        print(
            word,
            round(feature_weights[word], 4)
        )
    else:
        print(
            word,
            "not in vocabulary"
        )

not -2.6322
no 0.1192
never -0.239
great 2.4725
good 1.103
bad -0.9154
terrible -1.3449
excellent 1.6144
disappointed -0.7009
waste 0.0


In [101]:
test_reviews = [
    "great",
    "not great",
    "good",
    "not good",
    "bad",
    "not bad",
    "excellent product",
    "not an excellent product",
    "terrible product",
    "this product is terrible but this review explains exactly why"
]

for review in test_reviews:

    prediction = model.predict([review])[0]

    probability = model.predict_proba(
        [review]
    )[0]

    print(
        f"{review:60} "
        f"Prediction: {prediction} "
        f"Helpful probability: {probability[1]:.3f}"
    )

great                                                        Prediction: 1 Helpful probability: 0.906
not great                                                    Prediction: 1 Helpful probability: 0.502
good                                                         Prediction: 1 Helpful probability: 0.711
not good                                                     Prediction: 0 Helpful probability: 0.253
bad                                                          Prediction: 0 Helpful probability: 0.246
not bad                                                      Prediction: 0 Helpful probability: 0.101
excellent product                                            Prediction: 1 Helpful probability: 0.703
not an excellent product                                     Prediction: 0 Helpful probability: 0.445
terrible product                                             Prediction: 0 Helpful probability: 0.159
this product is terrible but this review explains exactly why Prediction: 0 Helpfu

### TF-IDF Bigram Experiment

The initial TF-IDF representation uses individual words (unigrams). Error analysis showed that this representation struggles with contextual phrases involving negation, such as "not bad" and "not good", because individual word weights are evaluated separately.

To test whether short phrase information improves classification, an additional TF-IDF representation using both unigrams and bigrams is evaluated with the previously selected SGDClassifier configuration.

This experiment changes only the text representation while keeping the classifier configuration fixed.

In [103]:
tfidf_bigram_sgd = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=20000,
            stop_words=sorted(stop_words),
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "sgd",
        SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            penalty="l1",
            class_weight={0: 2.5, 1: 1},
            max_iter=2000,
            tol=1e-3,
            random_state=42,
            n_jobs=-1
        )
    )
])

In [105]:
tfidf_bigram_sgd.fit(
    X_train_text,
    y_train
)

print("TF-IDF bigram + SGD model trained.")

TF-IDF bigram + SGD model trained.


In [107]:
bigram_pred = tfidf_bigram_sgd.predict(
    X_test_text
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            bigram_pred
        ),
        4
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        bigram_pred,
        digits=3
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        bigram_pred
    )
)

Accuracy: 0.8056

Classification Report:
              precision    recall  f1-score   support

           0      0.383     0.359     0.371      6699
           1      0.880     0.890     0.885     35293

    accuracy                          0.806     41992
   macro avg      0.631     0.625     0.628     41992
weighted avg      0.801     0.806     0.803     41992

Confusion Matrix:
[[ 2405  4294]
 [ 3871 31422]]


In [109]:
test_reviews = [
    "great",
    "not great",
    "good",
    "not good",
    "bad",
    "not bad",
    "excellent product",
    "not an excellent product",
    "terrible product",
    "this product is terrible but this review explains exactly why"
]

for review in test_reviews:

    prediction = tfidf_bigram_sgd.predict(
        [review]
    )[0]

    probability = tfidf_bigram_sgd.predict_proba(
        [review]
    )[0]

    print(
        f"{review:60} "
        f"Prediction: {prediction} "
        f"Helpful probability: {probability[1]:.3f}"
    )

great                                                        Prediction: 1 Helpful probability: 0.976
not great                                                    Prediction: 0 Helpful probability: 0.234
good                                                         Prediction: 1 Helpful probability: 0.814
not good                                                     Prediction: 0 Helpful probability: 0.120
bad                                                          Prediction: 0 Helpful probability: 0.183
not bad                                                      Prediction: 0 Helpful probability: 0.230
excellent product                                            Prediction: 1 Helpful probability: 0.757
not an excellent product                                     Prediction: 1 Helpful probability: 0.656
terrible product                                             Prediction: 0 Helpful probability: 0.234
this product is terrible but this review explains exactly why Prediction: 0 Helpfu

In [111]:
# 5-fold robustness check for TF-IDF unigram + bigram model

bigram_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

bigram_scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "class0_f1": make_scorer(
        f1_score,
        pos_label=0
    ),
    "class0_precision": make_scorer(
        precision_score,
        pos_label=0
    ),
    "class0_recall": make_scorer(
        recall_score,
        pos_label=0
    )
}

bigram_cv_results = cross_validate(
    tfidf_bigram_sgd,
    X_text,
    y,
    cv=bigram_cv,
    scoring=bigram_scoring,
    n_jobs=-1
)

print("5-fold bigram cross-validation completed.")

5-fold bigram cross-validation completed.


In [113]:
bigram_cv_summary = pd.DataFrame({
    "Accuracy": bigram_cv_results["test_accuracy"],
    "Macro F1": bigram_cv_results["test_macro_f1"],
    "Class 0 Precision": bigram_cv_results["test_class0_precision"],
    "Class 0 Recall": bigram_cv_results["test_class0_recall"],
    "Class 0 F1": bigram_cv_results["test_class0_f1"]
})

print("Fold-by-Fold Results:\n")
print(bigram_cv_summary.round(3))

print("\nAverage 5-Fold Results:\n")

for column in bigram_cv_summary.columns:
    print(
        f"{column}: "
        f"{bigram_cv_summary[column].mean():.3f} "
        f"(± {bigram_cv_summary[column].std():.3f})"
    )

Fold-by-Fold Results:

   Accuracy  Macro F1  Class 0 Precision  Class 0 Recall  Class 0 F1
0     0.808     0.630              0.389           0.361       0.374
1     0.811     0.626              0.393           0.336       0.363
2     0.805     0.629              0.384           0.362       0.373
3     0.804     0.631              0.383           0.373       0.378
4     0.807     0.637              0.393           0.383       0.388

Average 5-Fold Results:

Accuracy: 0.807 (± 0.003)
Macro F1: 0.631 (± 0.004)
Class 0 Precision: 0.388 (± 0.005)
Class 0 Recall: 0.363 (± 0.018)
Class 0 F1: 0.375 (± 0.009)


In [115]:
# Train final production model on all cleaned reviews

tfidf_bigram_sgd.fit(
    X_text,
    y
)

print("Final bigram TF-IDF + SGD model trained.")
print("Training rows:", len(y))

Final bigram TF-IDF + SGD model trained.
Training rows: 209960


In [117]:
import joblib

final_model_filename = (
    "amazon_helpfulness_tfidf_bigram_sgd.joblib"
)

joblib.dump(
    tfidf_bigram_sgd,
    final_model_filename,
    compress=3
)

print(
    "Final model saved as:",
    final_model_filename
)

Final model saved as: amazon_helpfulness_tfidf_bigram_sgd.joblib


In [119]:
loaded_bigram_model = joblib.load(
    "amazon_helpfulness_tfidf_bigram_sgd.joblib"
)

print("Bigram production model loaded successfully.")

Bigram production model loaded successfully.


In [121]:
print(
    "N-gram range:",
    loaded_bigram_model
    .named_steps["tfidf"]
    .ngram_range
)

print(
    "Max features:",
    loaded_bigram_model
    .named_steps["tfidf"]
    .max_features
)

print(
    "Minimum document frequency:",
    loaded_bigram_model
    .named_steps["tfidf"]
    .min_df
)

print(
    "SGD alpha:",
    loaded_bigram_model
    .named_steps["sgd"]
    .alpha
)

print(
    "SGD penalty:",
    loaded_bigram_model
    .named_steps["sgd"]
    .penalty
)

N-gram range: (1, 2)
Max features: 20000
Minimum document frequency: 2
SGD alpha: 1e-05
SGD penalty: l1


In [123]:
test_reviews = [
    "great",
    "not great",
    "good",
    "not good",
    "bad",
    "not bad",
    "excellent product",
    "not an excellent product",
    "terrible product"
]

for review in test_reviews:

    prediction = loaded_bigram_model.predict(
        [review]
    )[0]

    probability = (
        loaded_bigram_model
        .predict_proba([review])[0]
    )

    print(
        f"{review:35} "
        f"Prediction: {prediction} "
        f"Helpful probability: "
        f"{probability[1]:.3f}"
    )

great                               Prediction: 1 Helpful probability: 0.972
not great                           Prediction: 0 Helpful probability: 0.281
good                                Prediction: 1 Helpful probability: 0.870
not good                            Prediction: 0 Helpful probability: 0.156
bad                                 Prediction: 0 Helpful probability: 0.190
not bad                             Prediction: 0 Helpful probability: 0.231
excellent product                   Prediction: 1 Helpful probability: 0.766
not an excellent product            Prediction: 1 Helpful probability: 0.668
terrible product                    Prediction: 0 Helpful probability: 0.245
